# Resultados do TCC — Avaliação Automática de Respostas Curtas (pt-asag-2018)

Este notebook **não treina modelos**. Ele lê os CSVs gerados pelos scripts da pasta `scripts/` (salvos em `resultados/`) e reconstrói as tabelas do Capítulo 4 do TCC. Roda em poucos segundos e precisa apenas de `pandas`.

- `resultados/df_11/`: experimento principal (MiniLM e BERTimbau) e experimento comparativo restrito no particionamento **df_11** (questões inéditas no teste).
- `resultados/df_12/`: experimento principal no particionamento **df_12** (mesmas questões no treino e no teste), executado após a correção do Coh-Metrix.
- `resultados/verificacao/`: saída da verificação da coluna auxiliar `indice_original` no df_11.

Todas as métricas (MAE, RMSE e R²) são calculadas no **conjunto de teste**. As colunas `*_val` correspondem à validação interna feita dentro do conjunto de treinamento.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 60)

RES = Path("..") / "resultados"

def ler(caminho, embedding):
    df = pd.read_csv(RES / caminho)
    df["Embedding"] = embedding
    return df

df11 = pd.concat([ler("df_11/resultados_df_11.csv", "MiniLM"),
                  ler("df_11/resultados_df_11_bertimbau.csv", "BERTimbau")], ignore_index=True)
df12 = pd.concat([ler("df_12/resultados_df_12.csv", "MiniLM"),
                  ler("df_12/resultados_df_12_bertimbau.csv", "BERTimbau")], ignore_index=True)
restrito11 = pd.concat([ler("df_11/restrito_minilm_df_11.csv", "MiniLM"),
                        ler("df_11/restrito_bertimbau_df_11.csv", "BERTimbau")], ignore_index=True)

print("df_11:", df11.shape, "| df_12:", df12.shape, "| restrito df_11:", restrito11.shape)

df_11: (160, 10) | df_12: (160, 10) | restrito df_11: (24, 10)


In [2]:
METRICAS = ["MAE", "RMSE", "R²"]

def linha(df, embedding, cenario, modelo, rotulo=None):
    r = df[(df.Embedding == embedding) & (df["Cenário"] == cenario) & (df.Modelo == modelo)]
    assert len(r) == 1, (embedding, cenario, modelo)
    r = r.iloc[0]
    return {"Embedding": embedding if "Emb" in cenario or "Embeddings" in cenario else "–",
            "Representação": rotulo or cenario, "Modelo": modelo,
            **{m: round(r[m], 4) for m in METRICAS}}

def tabela(linhas):
    return pd.DataFrame(linhas).reset_index(drop=True)

EMB, TE, TUDO = "Apenas Embeddings", "TF-IDF (processado) + Embeddings", "Tudo (TF-IDF processado + Coh-Metrix + Emb)"
CE, TC, COH = "Coh-Metrix + Embeddings", "TF-IDF (processado) + Coh-Metrix", "Apenas Coh-Metrix"
VARIANTES = ["Apenas TF-IDF (cru)", "Apenas TF-IDF (stopwords)",
             "Apenas TF-IDF (stopwords+lemma)", "Apenas TF-IDF (stopwords+lemma+POS)"]

## Tabelas 3 e 4 — Efeito do pré-processamento do TF-IDF (R² no teste)

As variantes de TF-IDF não usam *embeddings*; por isso, os valores são idênticos nos arquivos MiniLM e BERTimbau.

In [3]:
def ablacao(df):
    t = (df[(df.Embedding == "MiniLM") & df["Cenário"].isin(VARIANTES)]
         .pivot(index="Modelo", columns="Cenário", values="R²")[VARIANTES])
    t.columns = ["Cru", "Stopwords", "Stopwords + Lematização", "Stopwords + Lematização + POS"]
    t["ΔR² (POS – Cru)"] = t.iloc[:, 3] - t.iloc[:, 0]
    return t.round(4)

print("Tabela 3 — df_12"); display(ablacao(df12))
print("Tabela 4 — df_11"); display(ablacao(df11))

Tabela 3 — df_12


,Cru,Stopwords,Stopwords + Lematização,Stopwords + Lematização + POS,ΔR² (POS – Cru)
Modelo,,,,,
HistGradientBoosting,0.6293,0.6346,0.6427,0.6380,0.0087
KNN,0.5117,0.5267,0.5392,0.5267,0.0150
Random Forest,0.6217,0.6177,0.6421,0.6305,0.0088
Rede Neural (MLP),0.2740,0.4859,0.5098,0.5146,0.2406
Regressão Linear,0.5714,0.5943,0.6004,0.5826,0.0112
SVR,0.5942,0.5848,0.5907,0.5729,-0.0213
XGBoost,0.6120,0.6190,0.6419,0.6284,0.0164
Árvore de Decisão,0.3445,0.3751,0.4066,0.3488,0.0043


Tabela 4 — df_11


,Cru,Stopwords,Stopwords + Lematização,Stopwords + Lematização + POS,ΔR² (POS – Cru)
Modelo,,,,,
HistGradientBoosting,0.1716,0.1451,0.2141,0.2157,0.0440
KNN,-0.7537,-0.7267,-0.8348,-0.3890,0.3647
Random Forest,0.0152,0.1125,0.1368,0.1638,0.1485
Rede Neural (MLP),-0.8882,-1.0348,-1.1286,-1.5171,-0.6289
Regressão Linear,-1.2313,-0.4378,-0.3448,-0.4837,0.7476
SVR,-0.5002,-0.5012,-0.1346,-0.1939,0.3063
XGBoost,-0.0524,0.1222,0.2358,0.0431,0.0955
Árvore de Decisão,-0.3840,-0.8707,-0.3123,-0.8523,-0.4682


## Tabela 5 — Melhores resultados do experimento principal no df_12

In [4]:
t5 = tabela([
    linha(df12, "MiniLM", TUDO, "HistGradientBoosting", "Tudo: TF-IDF processado + Coh-Metrix + embeddings"),
    linha(df12, "MiniLM", TE, "HistGradientBoosting", "TF-IDF processado + embeddings"),
    linha(df12, "MiniLM", CE, "HistGradientBoosting", "Coh-Metrix + embeddings"),
    linha(df12, "MiniLM", EMB, "HistGradientBoosting", "Apenas embeddings"),
    linha(df12, "BERTimbau", EMB, "SVR", "Apenas embeddings"),
    linha(df12, "BERTimbau", CE, "SVR", "Coh-Metrix + embeddings"),
    linha(df12, "BERTimbau", TUDO, "HistGradientBoosting", "Tudo: TF-IDF processado + Coh-Metrix + embeddings"),
    linha(df12, "BERTimbau", TE, "HistGradientBoosting", "TF-IDF processado + embeddings"),
])
t5

,Embedding,Representação,Modelo,MAE,RMSE,R²
0,MiniLM,Tudo: TF-IDF processado + Coh-Metrix + embeddings,HistGradientBoosting,0.4278,0.6226,0.6538
1,MiniLM,TF-IDF processado + embeddings,HistGradientBoosting,0.4263,0.6233,0.6531
2,MiniLM,Coh-Metrix + embeddings,HistGradientBoosting,0.4379,0.6338,0.6412
3,MiniLM,Apenas embeddings,HistGradientBoosting,0.4382,0.6374,0.6371
4,BERTimbau,Apenas embeddings,SVR,0.4187,0.6173,0.6597
5,BERTimbau,Coh-Metrix + embeddings,SVR,0.4184,0.6180,0.6589
6,BERTimbau,Tudo: TF-IDF processado + Coh-Metrix + embeddings,HistGradientBoosting,0.4329,0.6275,0.6483
7,BERTimbau,TF-IDF processado + embeddings,HistGradientBoosting,0.4347,0.6312,0.6441


## Tabela 6 — Melhores resultados do experimento principal no df_11

Linhas marcadas com `*` usaram o vetor do Coh-Metrix que incluía a coluna auxiliar `indice_original` (posição do registro no arquivo). A verificação ao final deste notebook mostra o efeito de removê-la.

In [5]:
t6 = tabela([
    linha(df11, "MiniLM", TC, "HistGradientBoosting", "TF-IDF processado + Coh-Metrix*"),
    linha(df11, "BERTimbau", EMB, "SVR", "Apenas embeddings"),
    linha(df11, "MiniLM", VARIANTES[2], "XGBoost", "Apenas TF-IDF com stopwords + lematização"),
    linha(df11, "MiniLM", VARIANTES[3], "HistGradientBoosting", "Apenas TF-IDF com stopwords + lematização + POS"),
    linha(df11, "MiniLM", VARIANTES[2], "HistGradientBoosting", "Apenas TF-IDF com stopwords + lematização"),
    linha(df11, "MiniLM", VARIANTES[0], "HistGradientBoosting", "Apenas TF-IDF cru"),
    linha(df11, "MiniLM", CE, "HistGradientBoosting", "Coh-Metrix + embeddings*"),
    linha(df11, "MiniLM", EMB, "SVR", "Apenas embeddings"),
])
t6

,Embedding,Representação,Modelo,MAE,RMSE,R²
0,–,TF-IDF processado + Coh-Metrix*,HistGradientBoosting,0.5910,0.7653,0.2619
1,BERTimbau,Apenas embeddings,SVR,0.5458,0.7757,0.2416
2,–,Apenas TF-IDF com stopwords + lematização,XGBoost,0.5866,0.7787,0.2358
3,–,Apenas TF-IDF com stopwords + lematização + POS,HistGradientBoosting,0.6120,0.7889,0.2157
4,–,Apenas TF-IDF com stopwords + lematização,HistGradientBoosting,0.5834,0.7897,0.2141
5,–,Apenas TF-IDF cru,HistGradientBoosting,0.5883,0.8107,0.1716
6,MiniLM,Coh-Metrix + embeddings*,HistGradientBoosting,0.6785,0.8121,0.1688
7,MiniLM,Apenas embeddings,SVR,0.5721,0.8295,0.1328


## Tabela 7 — Representações isoladas no df_11 (experimento comparativo restrito)

Experimento com SVR, Random Forest, Árvore de Decisão e XGBoost. O experimento restrito foi executado apenas para o df_11.

In [6]:
def restrito(embedding, features, model, rotulo, emb_col):
    r = restrito11[(restrito11.Embedding == embedding) & (restrito11.Features == features) & (restrito11.Model == model)].iloc[0]
    return {"Embedding": emb_col, "Representação isolada": rotulo, "Modelo": model,
            "MAE": r.MAE, "RMSE": r.RMSE, "R²": r.R2}

t7 = pd.DataFrame([
    restrito("MiniLM", "TFIDF", "RF", "TF-IDF", "–"),
    restrito("MiniLM", "Embeddings", "XGB", "Embeddings", "MiniLM"),
    restrito("MiniLM", "CohMetrix", "XGB", "Coh-Metrix*", "–"),
    restrito("BERTimbau", "Embeddings", "SVM", "Embeddings", "BERTimbau"),
]).replace({"Modelo": {"RF": "Random Forest", "XGB": "XGBoost", "SVM": "SVR"}})
t7

,Embedding,Representação isolada,Modelo,MAE,RMSE,R²
0,–,TF-IDF,Random Forest,0.6402,0.8145,0.1638
1,MiniLM,Embeddings,XGBoost,0.5960,0.8293,0.1331
2,–,Coh-Metrix*,XGBoost,0.7354,0.9345,-0.1007
3,BERTimbau,Embeddings,SVR,0.5458,0.7757,0.2416


## Tabela 8 — Representações isoladas no df_12

O experimento restrito não foi executado para o df_12; esta tabela usa os resultados do experimento principal, considerando todos os algoritmos.

In [7]:
t8 = tabela([
    linha(df12, "MiniLM", VARIANTES[2], "HistGradientBoosting", "TF-IDF"),
    linha(df12, "MiniLM", EMB, "HistGradientBoosting", "Embeddings"),
    linha(df12, "MiniLM", COH, "HistGradientBoosting", "Coh-Metrix"),
    linha(df12, "BERTimbau", EMB, "SVR", "Embeddings"),
])
t8

,Embedding,Representação,Modelo,MAE,RMSE,R²
0,–,TF-IDF,HistGradientBoosting,0.4330,0.6325,0.6427
1,MiniLM,Embeddings,HistGradientBoosting,0.4382,0.6374,0.6371
2,–,Coh-Metrix,HistGradientBoosting,0.4970,0.7046,0.5567
3,BERTimbau,Embeddings,SVR,0.4187,0.6173,0.6597


## Tabelas 9 e 10 — Comparação entre particionamentos e entre MiniLM e BERTimbau

In [8]:
def com_part(part, d):
    return {"Particionamento": part, **d}

t9 = pd.DataFrame([
    com_part("df_11", linha(df11, "BERTimbau", EMB, "SVR", "Apenas embeddings (BERTimbau)")),
    com_part("df_12", linha(df12, "BERTimbau", EMB, "SVR", "Apenas embeddings (BERTimbau)")),
]).drop(columns="Embedding")

t10 = pd.DataFrame([
    com_part("df_11", linha(df11, "MiniLM", EMB, "SVR", "Apenas embeddings")),
    com_part("df_11", linha(df11, "BERTimbau", EMB, "SVR", "Apenas embeddings")),
    com_part("df_12", linha(df12, "MiniLM", TUDO, "HistGradientBoosting", "TF-IDF processado + Coh-Metrix + embeddings")),
    com_part("df_12", linha(df12, "BERTimbau", EMB, "SVR", "Apenas embeddings")),
])
print("Tabela 9"); display(t9)
print("Tabela 10"); display(t10)

Tabela 9


,Particionamento,Representação,Modelo,MAE,RMSE,R²
0,df_11,Apenas embeddings (BERTimbau),SVR,0.5458,0.7757,0.2416
1,df_12,Apenas embeddings (BERTimbau),SVR,0.4187,0.6173,0.6597


Tabela 10


,Particionamento,Embedding,Representação,Modelo,MAE,RMSE,R²
0,df_11,MiniLM,Apenas embeddings,SVR,0.5721,0.8295,0.1328
1,df_11,BERTimbau,Apenas embeddings,SVR,0.5458,0.7757,0.2416
2,df_12,MiniLM,TF-IDF processado + Coh-Metrix + embeddings,HistGradientBoosting,0.4278,0.6226,0.6538
3,df_12,BERTimbau,Apenas embeddings,SVR,0.4187,0.6173,0.6597


## Ranking completo (para consulta)

As 15 configurações com maior R² no teste em cada particionamento (os cenários sem *embeddings* aparecem uma vez só).

In [9]:
def ranking(df, n=15):
    d = df.copy()
    # Cenários sem embeddings são idênticos nos arquivos MiniLM e BERTimbau: mostra uma vez só.
    d.loc[~d["Cenário"].str.contains("Emb"), "Embedding"] = "–"
    d = d.drop_duplicates(["Embedding", "Cenário", "Modelo"])
    cols = ["Embedding", "Cenário", "Modelo", "MAE", "RMSE", "R²", "R²_val"]
    return d.sort_values("R²", ascending=False)[cols].head(n).reset_index(drop=True)

print("df_11 — 15 melhores (inclui cenários com Coh-Metrix afetados pela coluna auxiliar)")
display(ranking(df11))
print("df_12 — 15 melhores")
display(ranking(df12))

df_11 — 15 melhores (inclui cenários com Coh-Metrix afetados pela coluna auxiliar)


,Embedding,Cenário,Modelo,MAE,RMSE,R²,R²_val
0,–,TF-IDF (processado) + Coh-Metrix,HistGradientBoosting,0.5910,0.7653,0.2619,-0.1385
1,BERTimbau,Apenas Embeddings,SVR,0.5458,0.7757,0.2416,-0.3210
2,–,Apenas TF-IDF (stopwords+lemma),XGBoost,0.5866,0.7787,0.2358,0.0152
3,BERTimbau,Coh-Metrix + Embeddings,SVR,0.5565,0.7846,0.2241,-0.2503
4,–,Apenas TF-IDF (stopwords+lemma+POS),HistGradientBoosting,0.6120,0.7889,0.2157,-0.3210
5,–,Apenas TF-IDF (stopwords+lemma),HistGradientBoosting,0.5834,0.7897,0.2141,-0.3894
6,–,Apenas TF-IDF (cru),HistGradientBoosting,0.5883,0.8107,0.1716,0.1077
7,MiniLM,TF-IDF (processado) + Embeddings,HistGradientBoosting,0.6803,0.8114,0.1702,-0.4483
8,MiniLM,Coh-Metrix + Embeddings,HistGradientBoosting,0.6785,0.8121,0.1688,-0.4655
9,–,Apenas TF-IDF (stopwords+lemma+POS),Random Forest,0.6402,0.8145,0.1638,-0.2683


df_12 — 15 melhores


,Embedding,Cenário,Modelo,MAE,RMSE,R²,R²_val
0,BERTimbau,Apenas Embeddings,SVR,0.4187,0.6173,0.6597,0.6636
1,BERTimbau,Coh-Metrix + Embeddings,SVR,0.4184,0.6180,0.6589,0.6599
2,MiniLM,Tudo (TF-IDF processado + Coh-Metrix + Emb),HistGradientBoosting,0.4278,0.6226,0.6538,0.6412
3,MiniLM,TF-IDF (processado) + Embeddings,HistGradientBoosting,0.4263,0.6233,0.6531,0.6455
4,BERTimbau,Tudo (TF-IDF processado + Coh-Metrix + Emb),HistGradientBoosting,0.4329,0.6275,0.6483,0.6325
5,BERTimbau,TF-IDF (processado) + Embeddings,HistGradientBoosting,0.4347,0.6312,0.6441,0.6439
6,–,Apenas TF-IDF (stopwords+lemma),HistGradientBoosting,0.4330,0.6325,0.6427,0.6225
7,–,Apenas TF-IDF (stopwords+lemma),Random Forest,0.4144,0.6330,0.6421,0.6082
8,–,Apenas TF-IDF (stopwords+lemma),XGBoost,0.4371,0.6333,0.6419,0.6048
9,MiniLM,Coh-Metrix + Embeddings,HistGradientBoosting,0.4379,0.6338,0.6412,0.6222


## Verificação da coluna auxiliar `indice_original` (df_11)

O script `scripts/verificar_indice_original_df11.py` reproduz os cenários com Coh-Metrix do df_11 com e sem essa coluna, usando os mesmos dados, código e parâmetros. A saída da execução está em `resultados/verificacao/`. A linha **COM** reproduz exatamente o valor registrado (R² 0,2619); sem a coluna, o R² cai para 0,0314.

In [10]:
saida = (RES / "verificacao" / "saida_verificacao_indice_original_df11.txt").read_text(encoding="utf-8")
print("\n".join(l for l in saida.splitlines() if "HistGradientBoosting" in l or l.startswith("===")))

Apenas TF-IDF (stopwords+lemma+POS)          HistGradientBoosting   MAE=0.6120 RMSE=0.7889 R2=0.2157
=== COM indice_original ===
Apenas Coh-Metrix [COM indice_original]      HistGradientBoosting   MAE=0.7392 RMSE=0.8826 R2=0.0182
TF-IDF + Coh-Metrix [COM indice_original]    HistGradientBoosting   MAE=0.5910 RMSE=0.7653 R2=0.2619
=== SEM indice_original ===
Apenas Coh-Metrix [SEM indice_original]      HistGradientBoosting   MAE=0.8889 RMSE=1.0302 R2=-0.3375
TF-IDF + Coh-Metrix [SEM indice_original]    HistGradientBoosting   MAE=0.7191 RMSE=0.8767 R2=0.0314
